# Week 1 — Complete Data Understanding & Quality Assessment
## Heavy Supplier, Inventory & Warehouse Analytics

**Purpose:** Run one notebook to complete the main Week 1 tasks:
1. Discover and load CSV datasets
2. Create a dataset inventory
3. Profile columns and missing values
4. Check duplicate rows and candidate keys
5. Inspect likely relationships between tables
6. Run cautious data-quality checks
7. Export reports for documentation

**Important:** This notebook does not modify or overwrite the original CSV files. Findings are signals to investigate, not automatic proof that a record is wrong.

### Before you start
Recommended project structure:

```text
heavy-supplier-analytics/
├── data/                   # original CSV files
├── week-01/
│   └── Week_1_Complete_Data_Profiling.ipynb
└── reports/                # generated CSV reports
```

Run the notebook from the `notebooks` folder if using the structure above. If the data folder has another name/location, update `DATA_DIR` in the setup cell.

## 1. Imports and configuration

Run this cell first. If a package is missing, install it in your active Python environment with `python -m pip install pandas numpy`.

In [4]:
from pathlib import Path
import pandas as pd
import numpy as np
import json
import re
from datetime import datetime

# Expected location when this notebook is inside the notebooks/ folder:
DATA_DIR = Path("../data")

# Folder where the notebook will save reports:
REPORT_DIR = Path("../reports")
REPORT_DIR.mkdir(parents=True, exist_ok=True)

# Set to True if you want to print more detailed information:
VERBOSE = True

print("Data folder:", DATA_DIR.resolve())
print("Reports folder:", REPORT_DIR.resolve())

Data folder: /Users/macbook/Documents/Heavy-Supplier-Inventory-Warehouse-Analytics/week-01/data
Reports folder: /Users/macbook/Documents/Heavy-Supplier-Inventory-Warehouse-Analytics/week-01/reports


## 2. Discover CSV files

This checks whether the configured data folder exists and lists CSV files found there.
If no files appear, correct `DATA_DIR` before continuing.

In [5]:
if not DATA_DIR.exists():
    raise FileNotFoundError(
        f"Data folder not found: {DATA_DIR.resolve()}. "
        "Update DATA_DIR to the folder containing your CSV files."
    )

csv_files = sorted(DATA_DIR.glob("*.csv"))

if not csv_files:
    raise FileNotFoundError(
        f"No CSV files found in {DATA_DIR.resolve()}. "
        "Check that the ZIP archive was extracted and DATA_DIR is correct."
    )

print(f"Found {len(csv_files)} CSV file(s):")
for file in csv_files:
    print(" -", file.name)

Found 12 CSV file(s):
 - branches.csv
 - customers.csv
 - inventory_master.csv
 - invoices.csv
 - payments.csv
 - products.csv
 - purchase_orders_header.csv
 - purchase_orders_lines.csv
 - sales_orders_header.csv
 - sales_orders_lines.csv
 - stock_ledger.csv
 - suppliers.csv


## 3. Load datasets safely

The dictionary key is the filename without `.csv`. For example, `products.csv` becomes `datasets["products"]`.

`low_memory=False` helps Pandas infer types using the full file rather than smaller chunks. It may use more memory.

In [6]:
datasets = {}

for file in csv_files:
    dataset_name = file.stem
    try:
        datasets[dataset_name] = pd.read_csv(file, low_memory=False)
        print(f"Loaded {file.name}: {datasets[dataset_name].shape[0]:,} rows, "
              f"{datasets[dataset_name].shape[1]:,} columns")
    except Exception as exc:
        print(f"Could not load {file.name}: {type(exc).__name__}: {exc}")

if not datasets:
    raise RuntimeError("No datasets were loaded successfully.")

print("\nDatasets loaded:", len(datasets))

Loaded branches.csv: 6 rows, 14 columns
Loaded customers.csv: 500 rows, 15 columns
Loaded inventory_master.csv: 180 rows, 8 columns
Loaded invoices.csv: 18,033 rows, 10 columns
Loaded payments.csv: 19,257 rows, 5 columns
Loaded products.csv: 30 rows, 23 columns
Loaded purchase_orders_header.csv: 24,000 rows, 10 columns
Loaded purchase_orders_lines.csv: 155,495 rows, 9 columns
Loaded sales_orders_header.csv: 20,000 rows, 11 columns
Loaded sales_orders_lines.csv: 130,402 rows, 9 columns
Loaded stock_ledger.csv: 237,230 rows, 9 columns
Loaded suppliers.csv: 8 rows, 12 columns

Datasets loaded: 12


## 4. Dataset inventory

Creates one summary row per loaded dataset: number of rows/columns, fully duplicated rows, missing cells, and approximate memory use.

In [7]:
inventory_rows = []

for name, df in datasets.items():
    inventory_rows.append({
        "dataset": name,
        "rows": len(df),
        "columns": len(df.columns),
        "duplicate_full_rows": int(df.duplicated().sum()),
        "missing_cells": int(df.isna().sum().sum()),
        "columns_with_missing_values": int(df.isna().any().sum()),
        "memory_mb": round(df.memory_usage(deep=True).sum() / (1024 ** 2), 2),
    })

data_inventory = pd.DataFrame(inventory_rows).sort_values(
    "rows", ascending=False
).reset_index(drop=True)

display(data_inventory)
data_inventory.to_csv(REPORT_DIR / "data_inventory.csv", index=False)
print("Saved:", (REPORT_DIR / "data_inventory.csv").resolve())

,dataset,rows,columns,duplicate_full_rows,missing_cells,columns_with_missing_values,memory_mb
0,stock_ledger,237230,9,0,0,0,91.76
1,purchase_orders_lines,155495,9,0,0,0,24.76
2,sales_orders_lines,130402,9,0,0,0,20.77
3,purchase_orders_header,24000,10,0,2370,1,9.71
4,sales_orders_header,20000,11,0,0,0,9.19
5,payments,19257,5,0,0,0,4.43
6,invoices,18033,10,0,0,0,7.28
7,customers,500,15,0,0,0,0.29
8,inventory_master,180,8,0,0,0,0.03
9,products,30,23,0,0,0,0.02


Saved: /Users/macbook/Documents/Heavy-Supplier-Inventory-Warehouse-Analytics/week-01/reports/data_inventory.csv


## 5. Column-level data profile

For every column, calculates data type, non-missing count, missing count and percentage, distinct values, and example values.

In [8]:
profile_rows = []

for dataset_name, df in datasets.items():
    for column in df.columns:
        series = df[column]
        examples = series.dropna().astype(str).drop_duplicates().head(3).tolist()

        profile_rows.append({
            "dataset": dataset_name,
            "column": column,
            "data_type": str(series.dtype),
            "non_null_count": int(series.notna().sum()),
            "missing_count": int(series.isna().sum()),
            "missing_percent": round(series.isna().mean() * 100, 2),
            "unique_non_null_values": int(series.nunique(dropna=True)),
            "example_values": " | ".join(examples),
        })

column_profile = pd.DataFrame(profile_rows)
display(column_profile.head(30))
column_profile.to_csv(REPORT_DIR / "column_profile.csv", index=False)
print("Saved:", (REPORT_DIR / "column_profile.csv").resolve())

,dataset,column,data_type,non_null_count,missing_count,missing_percent,unique_non_null_values,example_values
0,branches,branch_id,str,6,0,0.0,6,DEL001 | PUN001 | CHN001
1,branches,branch_name,str,6,0,0.0,6,Delhi Central | Pune Distribution | Chennai So...
2,branches,city,str,6,0,0.0,6,Delhi | Pune | Chennai
3,branches,state,str,6,0,0.0,6,Delhi | Maharashtra | Tamil Nadu
4,branches,region,str,6,0,0.0,4,North | West | South
5,branches,warehouse_type,str,6,0,0.0,4,Central | Regional | Branch
6,branches,warehouse_capacity,str,6,0,0.0,6,45230 sqft | 38750 sqft | 41890 sqft
7,branches,service_center_available,str,6,0,0.0,2,Yes | No
8,branches,manager_id,int64,6,0,0.0,6,9593 | 6114 | 7922
9,branches,total_employees,int64,6,0,0.0,6,92 | 74 | 81


Saved: /Users/macbook/Documents/Heavy-Supplier-Inventory-Warehouse-Analytics/week-01/reports/column_profile.csv


## 6. Review one dataset in detail

Change `TABLE_TO_REVIEW` to any dataset key printed in Section 3. This cell displays column names, data types, a preview, and descriptive statistics.

In [9]:
print("Available datasets:", list(datasets.keys()))

TABLE_TO_REVIEW = "products"  # Change this if needed

if TABLE_TO_REVIEW not in datasets:
    print(f"{TABLE_TO_REVIEW!r} was not found. Choose a name from Available datasets.")
else:
    df_review = datasets[TABLE_TO_REVIEW]
    print("Shape:", df_review.shape)
    print("\nColumns:")
    print(df_review.columns.tolist())
    print("\nFirst five rows:")
    display(df_review.head())
    print("\nDataFrame info:")
    df_review.info()
    print("\nDescriptive statistics (including text columns):")
    display(df_review.describe(include="all").T)

Available datasets: ['branches', 'customers', 'inventory_master', 'invoices', 'payments', 'products', 'purchase_orders_header', 'purchase_orders_lines', 'sales_orders_header', 'sales_orders_lines', 'stock_ledger', 'suppliers']
Shape: (30, 23)

Columns:
['product_id', 'product_name', 'category', 'machine_type', 'brand', 'model_compatibility', 'unit_cost', 'unit_price', 'margin_percentage', 'gst_rate', 'weight_kg', 'dimensions_cm', 'material_type', 'warranty_months', 'reorder_level', 'safety_stock', 'max_stock_level', 'lead_time_days', 'criticality_level', 'usage_frequency', 'uom', 'last_purchase_price', 'last_purchase_date']

First five rows:


,product_id,product_name,category,machine_type,brand,model_compatibility,unit_cost,unit_price,margin_percentage,gst_rate,...,warranty_months,reorder_level,safety_stock,max_stock_level,lead_time_days,criticality_level,usage_frequency,uom,last_purchase_price,last_purchase_date
0,P001,Hydraulic Pump HP-300,Hydraulic,Excavator,CAT,CAT 320D,18950,24500,29.2,28,...,12,15,10,40,21,High,Medium,piece,18500,2019-03-15
1,P002,Engine Oil Filter OF-90,Filters,Loader,JCB,JCB 3DX,450,720,37.3,18,...,6,80,40,200,7,Medium,High,piece,430,2019-07-22
2,P003,Air Filter AF-120,Filters,Excavator,Komatsu,Komatsu PC210,820,1250,52.4,18,...,6,60,30,150,7,Medium,High,piece,800,2020-01-10
3,P004,Track Chain TC-45,Undercarriage,Bulldozer,Volvo,Volvo D6,32500,39800,22.5,28,...,18,8,4,20,30,High,Low,set,32000,2020-06-18
4,P005,Fuel Injector FI-220,Engine,Excavator,Tata Hitachi,Tata Hitachi EX 200,6350,8450,33.0,28,...,12,30,15,80,18,High,Medium,piece,6200,2020-11-05



DataFrame info:
<class 'pandas.DataFrame'>
RangeIndex: 30 entries, 0 to 29
Data columns (total 23 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   product_id           30 non-null     str    
 1   product_name         30 non-null     str    
 2   category             30 non-null     str    
 3   machine_type         30 non-null     str    
 4   brand                30 non-null     str    
 5   model_compatibility  30 non-null     str    
 6   unit_cost            30 non-null     int64  
 7   unit_price           30 non-null     int64  
 8   margin_percentage    30 non-null     float64
 9   gst_rate             30 non-null     int64  
 10  weight_kg            30 non-null     float64
 11  dimensions_cm        30 non-null     str    
 12  material_type        30 non-null     str    
 13  warranty_months      30 non-null     int64  
 14  reorder_level        30 non-null     int64  
 15  safety_stock         30 non-null    

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
product_id,30,30,P001,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
product_name,30,30,Hydraulic Pump HP-300,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
category,30,15,Hydraulic,6,NaN,NaN,NaN,NaN,NaN,NaN,NaN
machine_type,30,6,Excavator,16,NaN,NaN,NaN,NaN,NaN,NaN,NaN
brand,30,6,CAT,7,NaN,NaN,NaN,NaN,NaN,NaN,NaN
model_compatibility,30,18,JCB 3DX,5,NaN,NaN,NaN,NaN,NaN,NaN,NaN
unit_cost,30.0,NaN,NaN,NaN,14585.0,21478.824014,180.0,957.5,7600.0,18312.5,95500.0
unit_price,30.0,NaN,NaN,NaN,18593.666667,26542.40816,310.0,1450.0,10250.0,23850.0,119000.0
margin_percentage,30.0,NaN,NaN,NaN,40.666667,16.062149,20.8,29.575,36.7,51.3,84.6
gst_rate,30.0,NaN,NaN,NaN,23.666667,5.040069,18.0,18.0,28.0,28.0,28.0


## 7. Duplicate checks

This identifies fully duplicated rows across each table. It does not remove them.

In [10]:
duplicate_rows_report = pd.DataFrame([
    {
        "dataset": name,
        "rows": len(df),
        "fully_duplicated_rows": int(df.duplicated().sum()),
        "fully_duplicated_percent": round(df.duplicated().mean() * 100, 2) if len(df) else 0,
    }
    for name, df in datasets.items()
]).sort_values("fully_duplicated_rows", ascending=False)

display(duplicate_rows_report)
duplicate_rows_report.to_csv(REPORT_DIR / "duplicate_rows_report.csv", index=False)
print("Saved:", (REPORT_DIR / "duplicate_rows_report.csv").resolve())

,dataset,rows,fully_duplicated_rows,fully_duplicated_percent
0,branches,6,0,0.0
1,customers,500,0,0.0
2,inventory_master,180,0,0.0
3,invoices,18033,0,0.0
4,payments,19257,0,0.0
5,products,30,0,0.0
6,purchase_orders_header,24000,0,0.0
7,purchase_orders_lines,155495,0,0.0
8,sales_orders_header,20000,0,0.0
9,sales_orders_lines,130402,0,0.0


Saved: /Users/macbook/Documents/Heavy-Supplier-Inventory-Warehouse-Analytics/week-01/reports/duplicate_rows_report.csv


## 8. Candidate identifier checks

This section *suggests* possible ID columns based on their names, then tests missing and duplicate values. Naming patterns are only clues; confirm the true primary key from the documentation.

In [11]:
candidate_key_rows = []

for dataset_name, df in datasets.items():
    for column in df.columns:
        normalized = str(column).strip().lower()
        looks_like_id = (
            normalized == "id"
            or normalized.endswith("_id")
            or normalized.endswith("id")
            or normalized in {"code", "sku", "order_number", "invoice_number"}
        )
        if looks_like_id:
            candidate_key_rows.append({
                "dataset": dataset_name,
                "candidate_column": column,
                "row_count": len(df),
                "missing_values": int(df[column].isna().sum()),
                "duplicate_values_including_repeats": int(df[column].duplicated().sum()),
                "distinct_non_null_values": int(df[column].nunique(dropna=True)),
                "appears_unique_and_non_null": bool(
                    df[column].notna().all() and df[column].is_unique
                ),
            })

candidate_key_report = pd.DataFrame(candidate_key_rows)
if candidate_key_report.empty:
    print("No columns matched common identifier naming patterns.")
else:
    display(candidate_key_report)
    candidate_key_report.to_csv(REPORT_DIR / "candidate_key_report.csv", index=False)
    print("Saved:", (REPORT_DIR / "candidate_key_report.csv").resolve())

,dataset,candidate_column,row_count,missing_values,duplicate_values_including_repeats,distinct_non_null_values,appears_unique_and_non_null
0,branches,branch_id,6,0,0,6,True
1,branches,manager_id,6,0,0,6,True
2,customers,customer_id,500,0,0,500,True
3,customers,branch_id,500,0,494,6,False
4,inventory_master,product_id,180,0,150,30,False
5,inventory_master,branch_id,180,0,174,6,False
6,invoices,invoice_id,18033,0,197,17836,False
7,invoices,so_id,18033,0,0,18033,True
8,invoices,customer_id,18033,0,17533,500,False
9,invoices,branch_id,18033,0,18027,6,False


Saved: /Users/macbook/Documents/Heavy-Supplier-Inventory-Warehouse-Analytics/week-01/reports/candidate_key_report.csv


## 9. Relationship checks (configure verified column names)

Do not guess foreign-key columns. First inspect the column profile and documentation, then fill in `RELATIONSHIPS` below with real table and column names.

Each item describes a child table/column that should refer to a parent table/column. The check counts child values that do not appear in the parent key list. Blank child values are counted separately.

In [12]:
# EXAMPLE STRUCTURE ONLY — replace these values with verified column names.
# If you do not know the actual column names yet, leave the list empty.
RELATIONSHIPS = [
    {
        "child_table": "customers",
        "child_column": "branch_id",
        "parent_table": "branches",
        "parent_column": "branch_id",
    },
    {
        "child_table": "inventory_master",
        "child_column": "product_id",
        "parent_table": "products",
        "parent_column": "product_id",
    },
    {
        "child_table": "inventory_master",
        "child_column": "branch_id",
        "parent_table": "branches",
        "parent_column": "branch_id",
    },
    {
        "child_table": "purchase_orders_header",
        "child_column": "supplier_id",
        "parent_table": "suppliers",
        "parent_column": "supplier_id",
    },
    {
        "child_table": "purchase_orders_header",
        "child_column": "branch_id",
        "parent_table": "branches",
        "parent_column": "branch_id",
    },
    {
        "child_table": "purchase_orders_lines",
        "child_column": "po_id",
        "parent_table": "purchase_orders_header",
        "parent_column": "po_id",
    },
    {
        "child_table": "purchase_orders_lines",
        "child_column": "product_id",
        "parent_table": "products",
        "parent_column": "product_id",
    },
    {
        "child_table": "sales_orders_header",
        "child_column": "customer_id",
        "parent_table": "customers",
        "parent_column": "customer_id",
    },
    {
        "child_table": "sales_orders_header",
        "child_column": "branch_id",
        "parent_table": "branches",
        "parent_column": "branch_id",
    },
    {
        "child_table": "sales_orders_lines",
        "child_column": "so_id",
        "parent_table": "sales_orders_header",
        "parent_column": "so_id",
    },
    {
        "child_table": "sales_orders_lines",
        "child_column": "product_id",
        "parent_table": "products",
        "parent_column": "product_id",
    },
    {
        "child_table": "invoices",
        "child_column": "so_id",
        "parent_table": "sales_orders_header",
        "parent_column": "so_id",
    },
    {
        "child_table": "invoices",
        "child_column": "customer_id",
        "parent_table": "customers",
        "parent_column": "customer_id",
    },
    {
        "child_table": "invoices",
        "child_column": "branch_id",
        "parent_table": "branches",
        "parent_column": "branch_id",
    },
    {
        "child_table": "payments",
        "child_column": "invoice_id",
        "parent_table": "invoices",
        "parent_column": "invoice_id",
    },
    {
        "child_table": "stock_ledger",
        "child_column": "product_id",
        "parent_table": "products",
        "parent_column": "product_id",
    },
    {
        "child_table": "stock_ledger",
        "child_column": "branch_id",
        "parent_table": "branches",
        "parent_column": "branch_id",
    },
]

relationship_rows = []

for relationship in RELATIONSHIPS:
    child_table = relationship["child_table"]
    child_column = relationship["child_column"]
    parent_table = relationship["parent_table"]
    parent_column = relationship["parent_column"]

    required = [child_table, parent_table]
    if any(table not in datasets for table in required):
        print("Skipping relationship; table not found:", relationship)
        continue

    child_df = datasets[child_table]
    parent_df = datasets[parent_table]

    if child_column not in child_df.columns or parent_column not in parent_df.columns:
        print("Skipping relationship; column not found:", relationship)
        continue

    parent_values = parent_df[parent_column].dropna()
    child_values = child_df[child_column]
    non_null_child_values = child_values.dropna()
    unmatched_mask = ~non_null_child_values.isin(set(parent_values))

    relationship_rows.append({
        "child_table": child_table,
        "child_column": child_column,
        "parent_table": parent_table,
        "parent_column": parent_column,
        "child_rows": len(child_df),
        "missing_child_key_values": int(child_values.isna().sum()),
        "unmatched_non_null_child_values": int(unmatched_mask.sum()),
    })

relationship_report = pd.DataFrame(relationship_rows)
if relationship_report.empty:
    print("No relationships configured yet. Verify column names, then add them to RELATIONSHIPS.")
else:
    display(relationship_report)
    relationship_report.to_csv(REPORT_DIR / "relationship_report.csv", index=False)
    print("Saved:", (REPORT_DIR / "relationship_report.csv").resolve())

,child_table,child_column,parent_table,parent_column,child_rows,missing_child_key_values,unmatched_non_null_child_values
0,customers,branch_id,branches,branch_id,500,0,0
1,inventory_master,product_id,products,product_id,180,0,0
2,inventory_master,branch_id,branches,branch_id,180,0,0
3,purchase_orders_header,supplier_id,suppliers,supplier_id,24000,0,0
4,purchase_orders_header,branch_id,branches,branch_id,24000,0,0
5,purchase_orders_lines,po_id,purchase_orders_header,po_id,155495,0,0
6,purchase_orders_lines,product_id,products,product_id,155495,0,0
7,sales_orders_header,customer_id,customers,customer_id,20000,0,0
8,sales_orders_header,branch_id,branches,branch_id,20000,0,0
9,sales_orders_lines,so_id,sales_orders_header,so_id,130402,0,0


Saved: /Users/macbook/Documents/Heavy-Supplier-Inventory-Warehouse-Analytics/week-01/reports/relationship_report.csv


## 10. Inspect date-like and numeric columns

This is a *screening step*, not an automatic conversion or a final validation. It finds columns whose names suggest dates, quantities, costs, prices, amounts, or stock. Review the results before defining business rules.

In [13]:
date_name_pattern = re.compile(r"(^|_)(date|datetime|timestamp|time_stamp)($|_)",
    re.IGNORECASE)
numeric_name_pattern = re.compile(
    r"qty|quantity|price|cost|amount|total|stock|balance|weight|volume|rate|lead_time",
    re.IGNORECASE
)

date_candidates = []
numeric_candidates = []

for dataset_name, df in datasets.items():
    for column in df.columns:
        if date_name_pattern.search(str(column)):
            parsed = pd.to_datetime(df[column], errors="coerce")
            original_non_null = int(df[column].notna().sum())
            parse_failures = int((df[column].notna() & parsed.isna()).sum())
            date_candidates.append({
                "dataset": dataset_name,
                "column": column,
                "original_non_null_values": original_non_null,
                "unparseable_non_null_values": parse_failures,
            })

        if numeric_name_pattern.search(str(column)):
            converted = pd.to_numeric(df[column], errors="coerce")
            original_non_null = int(df[column].notna().sum())
            conversion_failures = int((df[column].notna() & converted.isna()).sum())
            numeric_candidates.append({
                "dataset": dataset_name,
                "column": column,
                "original_non_null_values": original_non_null,
                "non_numeric_non_null_values": conversion_failures,
                "minimum_numeric_value": converted.min(),
                "maximum_numeric_value": converted.max(),
            })

date_candidate_report = pd.DataFrame(date_candidates)
numeric_candidate_report = pd.DataFrame(numeric_candidates)

print("Date-like columns:")
display(date_candidate_report if not date_candidate_report.empty else "No date-like columns detected.")

print("Numeric-like columns:")
display(numeric_candidate_report if not numeric_candidate_report.empty else "No numeric-like columns detected.")

if not date_candidate_report.empty:
    date_candidate_report.to_csv(REPORT_DIR / "date_candidate_report.csv", index=False)
if not numeric_candidate_report.empty:
    numeric_candidate_report.to_csv(REPORT_DIR / "numeric_candidate_report.csv", index=False)

Date-like columns:


,dataset,column,original_non_null_values,unparseable_non_null_values
0,customers,last_purchase_date,500,0
1,invoices,invoice_date,18033,0
2,invoices,due_date,18033,0
3,payments,payment_date,19257,0
4,products,last_purchase_date,30,0
5,purchase_orders_header,order_date,24000,0
6,purchase_orders_header,expected_delivery_date,24000,0
7,purchase_orders_header,received_date,21630,0
8,sales_orders_header,order_date,20000,0
9,sales_orders_header,delivery_date,20000,0


Numeric-like columns:


,dataset,column,original_non_null_values,non_numeric_non_null_values,minimum_numeric_value,maximum_numeric_value
0,branches,total_employees,6,0,4.900000e+01,9.200000e+01
1,branches,monthly_operational_cost,6,0,1.743000e+06,3.189000e+06
2,customers,current_balance,500,0,6.300000e+01,1.435105e+06
3,customers,total_purchase_value,500,0,5.173000e+03,4.998438e+06
4,inventory_master,opening_stock,180,0,8.000000e+01,3.000000e+02
5,inventory_master,safety_stock,180,0,1.600000e+01,8.500000e+01
6,inventory_master,max_stock,180,0,1.310000e+02,5.820000e+02
7,inventory_master,current_stock,180,0,8.885300e+04,1.210130e+05
8,invoices,total_order_value,18033,0,3.100000e+02,8.169760e+06
9,invoices,total_gst_amount,18033,0,5.580000e+01,2.282222e+06


## 11. Create a cautious data-quality issue log

This consolidates selected warning signals into a single issue log. It does not decide automatically that the data is wrong; every finding needs interpretation.

In [14]:
issue_rows = []

for _, row in data_inventory.iterrows():
    if row["duplicate_full_rows"] > 0:
        issue_rows.append({
            "dataset": row["dataset"],
            "issue": "Fully duplicated rows detected",
            "count": int(row["duplicate_full_rows"]),
            "severity": "Review",
            "recommended_action": "Inspect sample records and confirm whether duplicates are legitimate before changing data.",
        })

    if row["missing_cells"] > 0:
        issue_rows.append({
            "dataset": row["dataset"],
            "issue": "Missing cells detected",
            "count": int(row["missing_cells"]),
            "severity": "Review",
            "recommended_action": "Use the column profile to identify affected fields and determine whether missingness is acceptable.",
        })

if not candidate_key_report.empty:
    for _, row in candidate_key_report.iterrows():
        if row["missing_values"] > 0:
            issue_rows.append({
                "dataset": row["dataset"],
                "issue": f"Missing values in candidate key: {row['candidate_column']}",
                "count": int(row["missing_values"]),
                "severity": "Investigate",
                "recommended_action": "Confirm whether this column is truly a primary key and investigate missing identifiers.",
            })
        if row["duplicate_values_including_repeats"] > 0:
            issue_rows.append({
                "dataset": row["dataset"],
                "issue": f"Repeated values in candidate key: {row['candidate_column']}",
                "count": int(row["duplicate_values_including_repeats"]),
                "severity": "Investigate",
                "recommended_action": "Confirm the table grain and whether this column is expected to be unique.",
            })

if not relationship_report.empty:
    for _, row in relationship_report.iterrows():
        if row["unmatched_non_null_child_values"] > 0:
            issue_rows.append({
                "dataset": row["child_table"],
                "issue": f"Unmatched references to {row['parent_table']}.{row['parent_column']}",
                "count": int(row["unmatched_non_null_child_values"]),
                "severity": "Investigate",
                "recommended_action": "Inspect unmatched values and verify the relationship and source-system rules.",
            })

quality_issue_log = pd.DataFrame(issue_rows, columns=[
    "dataset", "issue", "count", "severity", "recommended_action"
])

display(quality_issue_log if not quality_issue_log.empty else
        pd.DataFrame([{"result": "No issues were flagged by the checks that ran. This does not prove the data is error-free."}]))

quality_issue_log.to_csv(REPORT_DIR / "data_quality_issue_log.csv", index=False)
print("Saved:", (REPORT_DIR / "data_quality_issue_log.csv").resolve())

,dataset,issue,count,severity,recommended_action
0,purchase_orders_header,Missing cells detected,2370,Review,Use the column profile to identify affected fi...
1,customers,Repeated values in candidate key: branch_id,494,Investigate,Confirm the table grain and whether this colum...
2,inventory_master,Repeated values in candidate key: product_id,150,Investigate,Confirm the table grain and whether this colum...
3,inventory_master,Repeated values in candidate key: branch_id,174,Investigate,Confirm the table grain and whether this colum...
4,invoices,Repeated values in candidate key: invoice_id,197,Investigate,Confirm the table grain and whether this colum...
5,invoices,Repeated values in candidate key: customer_id,17533,Investigate,Confirm the table grain and whether this colum...
6,invoices,Repeated values in candidate key: branch_id,18027,Investigate,Confirm the table grain and whether this colum...
7,payments,Repeated values in candidate key: payment_id,202,Investigate,Confirm the table grain and whether this colum...
8,payments,Repeated values in candidate key: invoice_id,3221,Investigate,Confirm the table grain and whether this colum...
9,purchase_orders_header,Repeated values in candidate key: supplier_id,23992,Investigate,Confirm the table grain and whether this colum...


Saved: /Users/macbook/Documents/Heavy-Supplier-Inventory-Warehouse-Analytics/week-01/reports/data_quality_issue_log.csv


## 12. Week 1 completion checklist

Review these items before you consider Week 1 complete:

- [ ] All expected CSV files were discovered and loaded.
- [ ] `data_inventory.csv` was generated.
- [ ] `column_profile.csv` was generated and reviewed.
- [ ] Duplicate-row findings were investigated.
- [ ] Candidate identifiers were checked against the documentation.
- [ ] Relationships were configured using verified column names, then tested.
- [ ] Date and numeric screening results were reviewed.
- [ ] `data_quality_issue_log.csv` was reviewed and annotated.
- [ ] Original source files remain unchanged.
- [ ] Limitations and unresolved questions are documented.

### Important reporting principle
Report what the checks actually show. For example, say “17 repeated values were found in the candidate identifier column” rather than “17 incorrect records were found” unless you have evidence that they are errors.